In [2]:
import warnings
warnings.simplefilter(action='ignore',category=FutureWarning)


from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score
) 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("Telco_churn_data.xls")
df.head()
df.isnull().sum()
df.describe()
df.nunique()
df.head()
yes_no_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]
df[yes_no_columns]= df[yes_no_columns].replace({
    "Yes":1,
    "No":0
})
df["gender"] = df["gender"].replace({
    "Male":1,
    "Female":0
})
df.drop(columns="customerID",inplace = True)
df = pd.get_dummies(
    df,
    columns = [
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "PaymentMethod",
        "Contract"
    ],
    drop_first=True
)
pd.to_numeric(df["TotalCharges"],errors='coerce')
mask = pd.to_numeric(df["TotalCharges"],errors='coerce').isna()
df = df[~mask]
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])
#line below is meant to address skewness
df["TotalCharges"] = np.log1p(df["TotalCharges"]) 
cols_to_scale = ["tenure","MonthlyCharges","TotalCharges"]
scalr = StandardScaler()
df[cols_to_scale] = scalr.fit_transform(df[cols_to_scale])
df["TotalCharges"].head()
df = df.drop(
    columns=[
        'OnlineSecurity_No internet service',
        'DeviceProtection_No internet service',
        'StreamingTV_No internet service',
        'OnlineBackup_No internet service',
        'TechSupport_No internet service',
        'StreamingMovies_No internet service'
    ])

In [4]:
X = df.drop(columns=["Churn"],axis=1)
Y=df["Churn"]
X_train,X_test,Y_train,Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)
model = LogisticRegression(class_weight='balanced',C=0.01)
model.fit(X_train,Y_train)
prediction = model.predict(X_test)
accuracy = accuracy_score(Y_test,prediction)
cm = confusion_matrix(Y_test,prediction)
cr = classification_report(Y_test,prediction)
print(accuracy)
print(cm)
print(cr)


0.7420042643923241
[[752 281]
 [ 82 292]]
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1033
           1       0.51      0.78      0.62       374

    accuracy                           0.74      1407
   macro avg       0.71      0.75      0.71      1407
weighted avg       0.80      0.74      0.76      1407



In [65]:
probs = model.predict_proba(X_test)[:,1]
for threshold in [0.3,0.45,0.5,0.6,0.7]:
    pred = (probs>threshold).astype(int)
    pro_cm = confusion_matrix(Y_test,pred)
    pro_cr = classification_report(Y_test,pred)
    print(pro_cm)
    print(pro_cr)


[[585 448]
 [ 33 341]]
              precision    recall  f1-score   support

           0       0.95      0.57      0.71      1033
           1       0.43      0.91      0.59       374

    accuracy                           0.66      1407
   macro avg       0.69      0.74      0.65      1407
weighted avg       0.81      0.66      0.68      1407

[[694 339]
 [ 58 316]]
              precision    recall  f1-score   support

           0       0.92      0.67      0.78      1033
           1       0.48      0.84      0.61       374

    accuracy                           0.72      1407
   macro avg       0.70      0.76      0.70      1407
weighted avg       0.81      0.72      0.73      1407

[[725 308]
 [ 72 302]]
              precision    recall  f1-score   support

           0       0.91      0.70      0.79      1033
           1       0.50      0.81      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weigh

In [13]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_.flatten()
})


coef_df = coef_df.sort_values(
    by="Coefficient",
    ascending=False
)

print(coef_df)

                                  Feature  Coefficient
6                          MonthlyCharges     0.565436
10            InternetService_Fiber optic     0.466359
19         PaymentMethod_Electronic check     0.307030
5                        PaperlessBilling     0.255334
0                           SeniorCitizen     0.251391
8          MultipleLines_No phone service     0.188563
9                       MultipleLines_Yes     0.153493
17                    StreamingMovies_Yes     0.151583
16                        StreamingTV_Yes     0.144436
1                                 Partner    -0.004396
14                   DeviceProtection_Yes    -0.049027
13                       OnlineBackup_Yes    -0.074866
18  PaymentMethod_Credit card (automatic)    -0.096873
20             PaymentMethod_Mailed check    -0.132533
4                            PhoneService    -0.188958
2                              Dependents    -0.189725
12                     OnlineSecurity_Yes    -0.304498
15        

In [29]:
results = X_test.copy()
results["Actual"] = Y_test
results["Predicted"] = prediction
false_positives = results[
    (results["Actual"]==0) & 
    (results["Predicted"]==1)]
print(false_positives.head())

      SeniorCitizen  Partner  Dependents    tenure  PhoneService  \
3223              0        1           0 -1.198760             0   
3469              1        0           0 -0.709833             1   
2173              0        0           0 -1.158016             1   
3257              0        0           0 -0.424625             1   
4042              0        0           0 -1.117272             1   

      PaperlessBilling  MonthlyCharges  TotalCharges  \
3223                 0       -1.163356     -1.542563   
3469                 1        0.887579      0.195152   
2173                 0        0.706419     -0.782319   
3257                 1        0.944088      0.474472   
4042                 1       -0.447024     -0.886030   

      MultipleLines_No phone service  MultipleLines_Yes  ...  TechSupport_Yes  \
3223                            True              False  ...            False   
3469                           False              False  ...            False   
2173       

In [80]:
print(df['tenure'].corr(df['Contract_Two year']))

0.5638005002286525
